# IFRS 9 Expected Credit Loss (ECL) Framework
## Freddie Mac Mortgage Portfolio — Quantitative Risk Implementation

**Framework:** IFRS 9 Financial Instruments (IASB, effective 2018)  
**Pipeline:**
```
PD Model → LGD Estimation → EAD Estimation
       ↓
Macroeconomic Scenario Conditioning
       ↓
Stage Classification (1 / 2 / 3)
       ↓
ECL = PD × LGD × EAD  (12-month or Lifetime)
       ↓
Scenario Weighting → Final Provision
       ↓
Validation (KS, Brier, PSI, Calibration, Backtesting)
```

> **Note on scaled features:** The modelling data uses `RobustScaler`-transformed features.
> All economic interpretations are made relative to the training-set distribution.
> LGD and EAD proxies are constructed from available columns; a production system
> would use recovery cash-flow data and Credit Conversion Factors (CCFs).


## 0. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, roc_curve,
    mean_absolute_error, mean_squared_error
)
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

import joblib
np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("Imports done.")


Imports done.


## 1. Load Data

In [2]:
BASE = r"C:\\Users\\Niraj Mhatre\\projects\\Mortgage-Portfolio-Risk-Analytics-and-IFRS-9-Provisioning-Framework\\notebooks\\"

train_df = pd.read_csv(BASE + "train_df.csv")
val_df   = pd.read_csv(BASE + "val_df.csv")
test_df  = pd.read_csv(BASE + "test_df.csv")

TARGET = "defaulted_flag"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_val   = val_df.drop(columns=[TARGET])
y_val   = val_df[TARGET]
X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

# Full portfolio = train + val + test (for reporting)
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Default rates — Train: {y_train.mean():.2%} | Val: {y_val.mean():.2%} | Test: {y_test.mean():.2%}")
print(f"Total portfolio: {len(full_df):,} loans")


Train: 350,000 | Val: 50,000 | Test: 150,000
Default rates — Train: 3.18% | Val: 8.96% | Test: 2.12%
Total portfolio: 550,000 loans


## 2. PD Model — Probability of Default

### Methodology
Logistic Regression with `class_weight='balanced'` is used as the primary PD model
for IFRS 9 because:
- **Interpretable coefficients** → auditors and regulators can inspect log-odds
- **Output is a calibrated probability** → directly usable as PD in ECL formula
- **Stable under stress scenarios** → linear log-odds relationships hold more 
  predictably than tree splits when extrapolating to macro-stressed inputs

A Random Forest is also trained as a challenger model for comparison.

> **IFRS 9 requirement:** PD must be **forward-looking** and **point-in-time (PiT)**,
> not the through-the-cycle (TtC) PDs used in Basel II. PiT PDs are conditioned
> on the current economic environment.


In [ ]:
# ── Train primary PD model (Logistic Regression) ───────────────────────────
smote = SMOTE(sampling_strategy=0.2, random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

lr_pd = LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")
lr_pd.fit(X_train_sm, y_train_sm)

# ── Challenger: Random Forest ───────────────────────────────────────────────
rf_pd = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=100,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
rf_pd.fit(X_train, y_train)

print("PD models trained.")
print(f"LR   Val ROC-AUC: {roc_auc_score(y_val, lr_pd.predict_proba(X_val)[:,1]):.4f}")
print(f"RF   Val ROC-AUC: {roc_auc_score(y_val, rf_pd.predict_proba(X_val)[:,1]):.4f}")


### 2.1 PD Calibration

Raw model probabilities must be **calibrated** before use as IFRS 9 PDs.
A poorly calibrated model may output PD=2% when the observed default rate is 4% —
that would directly understate the ECL provision.

We use **Platt scaling** (logistic calibration) which is the industry standard
for scorecards and credit models. Isotonic regression is shown as an alternative.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# Platt scaling on val set
lr_calibrated = CalibratedClassifierCV(lr_pd, method="sigmoid", cv="prefit")
lr_calibrated.fit(X_val, y_val)

rf_calibrated = CalibratedClassifierCV(rf_pd, method="sigmoid", cv="prefit")
rf_calibrated.fit(X_val, y_val)

# ── Calibration curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, raw, cal) in zip(axes, [
    ("Logistic Regression", lr_pd, lr_calibrated),
    ("Random Forest",       rf_pd, rf_calibrated),
]):
    prob_raw = raw.predict_proba(X_val)[:, 1]
    prob_cal = cal.predict_proba(X_val)[:, 1]

    frac_raw, mean_raw = calibration_curve(y_val, prob_raw, n_bins=15)
    frac_cal, mean_cal = calibration_curve(y_val, prob_cal, n_bins=15)

    ax.plot([0,1],[0,1], "k--", lw=1, label="Perfect calibration")
    ax.plot(mean_raw, frac_raw, "o-", label=f"Raw  (Brier={brier_score_loss(y_val, prob_raw):.4f})")
    ax.plot(mean_cal, frac_cal, "s-", label=f"Platt (Brier={brier_score_loss(y_val, prob_cal):.4f})")
    ax.set_xlabel("Predicted PD"); ax.set_ylabel("Observed Default Rate")
    ax.set_title(f"{name} — Calibration Plot")
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("PD Calibration: Raw vs Platt-Scaled", fontsize=12)
plt.tight_layout(); plt.show()

# Use calibrated LR as primary PD model
PD_MODEL = lr_calibrated
print("Primary PD model: Logistic Regression + Platt calibration")
print(f"Brier score (val): {brier_score_loss(y_val, PD_MODEL.predict_proba(X_val)[:,1]):.6f}")


In [ ]:
# ── Generate point-in-time PDs for the full portfolio ─────────────────────
X_full = full_df.drop(columns=[TARGET])
y_full = full_df[TARGET]

full_df["PD_12m"]  = PD_MODEL.predict_proba(X_full)[:, 1]

# Lifetime PD approximation:
# For a fixed-rate mortgage, lifetime PD is derived from the 12-month PD
# using a constant hazard rate assumption:
#   Lifetime PD ≈ 1 - (1 - PD_12m)^N
# where N = remaining loan term in years (we proxy from fp_vintage scaling)
# Without exact term data we assume average remaining term = 15 years (Freddie Mac
# typical remaining life at origination), adjusted by a vintage factor.
AVG_REMAINING_YEARS = 15

full_df["PD_lifetime"] = 1 - (1 - full_df["PD_12m"]) ** AVG_REMAINING_YEARS
full_df["PD_lifetime"] = full_df["PD_lifetime"].clip(0, 1)

print(f"PD_12m  — mean: {full_df['PD_12m'].mean():.4f} | min: {full_df['PD_12m'].min():.4f} | max: {full_df['PD_12m'].max():.4f}")
print(f"PD_life — mean: {full_df['PD_lifetime'].mean():.4f}")


## 3. LGD — Loss Given Default

### Methodology
LGD is the fraction of EAD that is **not recovered** after a default event.
For residential mortgages:

```
LGD = 1 - Recovery Rate
Recovery Rate = Collateral Recovery + MI Recovery - Costs
```

**Inputs available in this dataset:**
- `og_ltv` (scaled): Loan-to-Value ratio — primary driver of collateral recovery
- `mortgage_insurance_percent`: MI coverage — directly offsets loss
- `cltv_ltv_ratio`: second-lien burden reduces net recovery

**Approach:**
1. **Proxy LGD** from economic formula using available features (regulatory approach)
2. **Model-based LGD** using a Gradient Boosting Regressor on defaulted loans

In a production setting, LGD would be estimated from observed recovery cash flows
discounted at the Effective Interest Rate (EIR). Here we use the proxy approach.

> **IFRS 9 requirement:** LGD must be **downturn LGD** — estimated under adverse
> economic conditions, not long-run average. We apply a downturn scalar below.


In [ ]:
# ── LGD Proxy Construction ─────────────────────────────────────────────────
# og_ltv is RobustScaler-transformed. We work with it directly.
# The economic formula:
#   LGD_base = max(0, LTV_factor) × (1 - MI_coverage_fraction)
#
# LTV_factor: higher (more positive) scaled LTV → more underwater → higher loss
# We map scaled og_ltv to a [0, 1] loss factor using a sigmoid:
#   loss_factor = sigmoid(og_ltv × 1.5)   (1.5 steepness chosen to match
#                                            ~25% LGD at median LTV and ~70% at high LTV)
#
# MI recovery: mortgage_insurance_percent is unscaled (0–55%)
#   MI_recovery = min(MI%, 30%) / 100   (cap at 30% since MI doesn't cover full loss)
#
# Downturn scalar = 1.15  (15% stress adder — typical bank practice for residential)
# Haircut for selling costs / legal = 10%

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

LTV_STEEPNESS  = 1.5
SELLING_COST   = 0.10
DOWNTURN_SCALE = 1.15
MI_CAP         = 0.30

for df in [train_df, val_df, test_df, full_df]:
    ltv_loss_factor = sigmoid(df["og_ltv"] * LTV_STEEPNESS)
    mi_recovery     = np.minimum(df["mortgage_insurance_percent"] / 100, MI_CAP)
    cltv_penalty    = np.abs(df["cltv_ltv_ratio"]).clip(0, 0.20)  # second-lien burden

    lgd_raw = (ltv_loss_factor - mi_recovery + cltv_penalty + SELLING_COST).clip(0, 1)
    df["LGD"] = (lgd_raw * DOWNTURN_SCALE).clip(0, 1)

print(f"LGD stats (full portfolio):")
print(full_df["LGD"].describe().round(4))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(full_df["LGD"], bins=60, edgecolor="white", color="crimson", alpha=0.8)
ax.axvline(full_df["LGD"].mean(),   color="black", lw=1.5, label=f"Mean  = {full_df['LGD'].mean():.3f}")
ax.axvline(full_df["LGD"].median(), color="navy",  lw=1.5, linestyle="--", label=f"Median = {full_df['LGD'].median():.3f}")
ax.set_xlabel("LGD"); ax.set_ylabel("Count")
ax.set_title("LGD Distribution — Full Portfolio (Downturn-Adjusted)")
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Model-based LGD (challenger) using defaulted loans ────────────────────
# On real data this would be trained on realised loss severities.
# Here we train on the proxy LGD to demonstrate the modelling workflow.
defaults_train = train_df[train_df[TARGET] == 1].copy()
X_lgd = defaults_train.drop(columns=[TARGET, "LGD"], errors="ignore")
y_lgd = defaults_train["LGD"]

lgd_model = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
lgd_model.fit(X_lgd, y_lgd)

# Apply to val defaults to check fit
defaults_val = val_df[val_df[TARGET] == 1].copy()
X_lgd_val    = defaults_val.drop(columns=[TARGET, "LGD"], errors="ignore")
lgd_pred_val = lgd_model.predict(X_lgd_val).clip(0, 1)

print(f"LGD Model — Val MAE : {mean_absolute_error(defaults_val['LGD'], lgd_pred_val):.4f}")
print(f"LGD Model — Val RMSE: {np.sqrt(mean_squared_error(defaults_val['LGD'], lgd_pred_val)):.4f}")
print("(Note: model trained on proxy LGD — in production, use realised severity)")


## 4. EAD — Exposure at Default

### Methodology
EAD is the outstanding balance at the time of default.

For **closed-end term mortgages** (Freddie Mac fixed-rate), EAD approximates the
outstanding principal balance because:
- There is no undrawn commitment (unlike revolving credit)
- Credit Conversion Factor (CCF) = 1.0 by convention

```
EAD ≈ Outstanding Principal Balance at default date
     ≈ og_upb × amortisation_factor(seasoning)
```

`og_upb` in this dataset is the **original** UPB (at origination), scaled.
We apply a simple amortisation factor to approximate the balance at time-of-default.

> **IFRS 9 requirement:** EAD must be estimated at the **expected time of default**,
> not at the reporting date. For Stage 1/2 this requires projecting the balance
> forward — here approximated using an average seasoning scalar.


In [ ]:
# ── EAD Construction ───────────────────────────────────────────────────────
# og_upb is RobustScaler-transformed. We interpret it as a relative balance.
# Amortisation factor: for a 30-year mortgage, roughly 20-30% of principal
# is repaid in the first 10 years. We use a constant 0.85 factor
# (approximate outstanding balance as 85% of original at mid-life).
# In production: use scheduled amortisation tables per loan.

AMORTISATION_FACTOR = 0.85   # outstanding balance as % of original UPB

for df in [train_df, val_df, test_df, full_df]:
    # EAD in scaled units (consistent with og_upb scale)
    df["EAD"] = df["og_upb"] * AMORTISATION_FACTOR

    # Floor EAD at 0 (can't have negative exposure)
    df["EAD"] = df["EAD"].clip(lower=0)

print(f"EAD stats (full portfolio):")
print(full_df["EAD"].describe().round(4))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(full_df["EAD"], bins=60, edgecolor="white", color="steelblue", alpha=0.8)
ax.axvline(full_df["EAD"].mean(), color="black", lw=1.5, label=f"Mean EAD = {full_df['EAD'].mean():.3f}")
ax.set_xlabel("EAD (scaled UPB units)"); ax.set_ylabel("Count")
ax.set_title("EAD Distribution — Full Portfolio")
ax.legend(); plt.tight_layout(); plt.show()


## 5. Macroeconomic Scenario Conditioning

### Why This Matters
IFRS 9 explicitly requires that ECL estimates incorporate **forward-looking
macroeconomic information**. A model trained on historical data produces
**through-the-cycle (TtC)** PDs. We must convert to **point-in-time (PiT)** PDs
conditioned on the economic outlook.

### Scenarios (IFRS 9 Standard)
Three scenarios are standard practice:

| Scenario | Economic Outlook | Weight |
|---|---|---|
| **Base** | Central / consensus forecast | 50% |
| **Adverse** | Moderate recession | 30% |
| **Severe** | Severe stress / tail risk | 20% |

### Macro Adjustment Mechanism
We use an **overlay scalar** applied to the base PD:
```
PD_scenario = PD_base × macro_scalar(scenario)
```

The macro scalars are calibrated to reflect typical mortgage default rate sensitivity
to economic stress. In a production system these would come from a macro-satellite
model (VAR, factor model) regressing historical default rates on GDP, unemployment,
and house price indices.

> **Source for scalar calibration**: Freddie Mac historical serious delinquency rates
> peaked at ~4.2% in 2010 vs ~0.3% in 2005 — a ~14× stress factor.
> Our scenarios are scaled proportionally.


In [ ]:
# ── Macroeconomic Scenario Scalars ─────────────────────────────────────────
SCENARIOS = {
    "Base":   {"pd_scalar": 1.00, "lgd_scalar": 1.00, "weight": 0.50},
    "Adverse":{"pd_scalar": 1.80, "lgd_scalar": 1.15, "weight": 0.30},
    "Severe": {"pd_scalar": 3.50, "lgd_scalar": 1.30, "weight": 0.20},
}

# Apply scenario scalars to the full portfolio
for scenario, params in SCENARIOS.items():
    pd_col  = f"PD_12m_{scenario}"
    pdl_col = f"PD_life_{scenario}"
    lgd_col = f"LGD_{scenario}"

    full_df[pd_col]  = (full_df["PD_12m"]      * params["pd_scalar"]).clip(0, 1)
    full_df[pdl_col] = (full_df["PD_lifetime"]  * params["pd_scalar"]).clip(0, 1)
    full_df[lgd_col] = (full_df["LGD"]          * params["lgd_scalar"]).clip(0, 1)

print("Scenario PD summary:")
for s in SCENARIOS:
    print(f"  {s:8s}: mean PD_12m = {full_df[f'PD_12m_{s}'].mean():.4f} | "
          f"mean LGD = {full_df[f'LGD_{s}'].mean():.4f}")


In [ ]:
# ── Visualise scenario PD distributions ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
colors = {"Base": "steelblue", "Adverse": "darkorange", "Severe": "crimson"}

for ax, scenario in zip(axes, SCENARIOS):
    col = f"PD_12m_{scenario}"
    ax.hist(full_df[col], bins=60, color=colors[scenario], edgecolor="white", alpha=0.85)
    ax.axvline(full_df[col].mean(), color="black", lw=1.5,
               label=f"Mean={full_df[col].mean():.3f}")
    ax.set_title(f"{scenario} Scenario
(scalar={SCENARIOS[scenario]['pd_scalar']}×, "
                 f"weight={SCENARIOS[scenario]['weight']:.0%})")
    ax.set_xlabel("PD (12-month)"); ax.legend()

plt.suptitle("PD Distribution by Macroeconomic Scenario", fontsize=12)
plt.tight_layout(); plt.show()


## 6. Stage Classification (SICR — Significant Increase in Credit Risk)

### IFRS 9 Staging Rules
| Stage | Condition | ECL Measure |
|---|---|---|
| **Stage 1** | No significant credit deterioration since origination | 12-month ECL |
| **Stage 2** | Significant Increase in Credit Risk (SICR) but not yet defaulted | Lifetime ECL |
| **Stage 3** | Credit-impaired (defaulted) | Lifetime ECL on impaired basis |

### SICR Criteria (Quantitative)
SICR triggers when the **absolute increase in PD** since origination or the
**relative ratio of current PD to origination PD** exceeds thresholds:

```
SICR if:  PD_current > PD_origination × RELATIVE_THRESHOLD
      or  PD_current - PD_origination > ABSOLUTE_THRESHOLD
      or  loan is delinquent (30+ DPD proxy)
```

We proxy origination-time PD using the vintage cohort median and
approximate SICR from current PD relativities within the portfolio.


In [ ]:
# ── Stage Classification ───────────────────────────────────────────────────
# Thresholds — typical bank practice:
PD_ABSOLUTE_THRESHOLD  = 0.02   # +2pp absolute increase triggers Stage 2
PD_RELATIVE_THRESHOLD  = 2.50   # 2.5× relative increase triggers Stage 2

# Origination-PD proxy: use the bottom-20th percentile PD in the cohort
# (approximates the PD at origination when loans were performing)
ORIGINATION_PD_PROXY = full_df["PD_12m"].quantile(0.20)

# Stage assignment
def assign_stage(row, orig_pd=ORIGINATION_PD_PROXY):
    if row[TARGET] == 1:
        return 3   # Stage 3: defaulted
    pd_now = row["PD_12m_Base"]
    abs_increase = pd_now - orig_pd
    rel_ratio    = pd_now / (orig_pd + 1e-9)
    if abs_increase > PD_ABSOLUTE_THRESHOLD or rel_ratio > PD_RELATIVE_THRESHOLD:
        return 2   # Stage 2: SICR
    return 1       # Stage 1: performing

full_df["Stage"] = full_df.apply(assign_stage, axis=1)

stage_counts = full_df["Stage"].value_counts().sort_index()
stage_pct    = (stage_counts / len(full_df) * 100).round(2)

print("Stage Distribution:")
for s in [1, 2, 3]:
    print(f"  Stage {s}: {stage_counts.get(s,0):8,} loans ({stage_pct.get(s,0):.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_stage = {1: "steelblue", 2: "darkorange", 3: "crimson"}

axes[0].bar(
    [f"Stage {s}" for s in [1,2,3]],
    [stage_counts.get(s,0) for s in [1,2,3]],
    color=[colors_stage[s] for s in [1,2,3]], edgecolor="white"
)
axes[0].set_title("Portfolio by IFRS 9 Stage"); axes[0].set_ylabel("Loan Count")

# Stage default rate
for s in [1,2,3]:
    sub = full_df[full_df["Stage"] == s]
    axes[1].bar(f"Stage {s}", sub[TARGET].mean() * 100, color=colors_stage[s], edgecolor="white")
axes[1].set_title("Observed Default Rate by Stage"); axes[1].set_ylabel("Default Rate (%)")
plt.suptitle("IFRS 9 Stage Allocation", fontsize=12)
plt.tight_layout(); plt.show()


## 7. ECL Calculation

### Formula
$$ECL_{loan} = PD \times LGD \times EAD$$

- **Stage 1**: Use **12-month PD**
- **Stage 2 & 3**: Use **Lifetime PD**

### Scenario-Weighted ECL
$$ECL_{final} = \sum_{s} w_s \times ECL_s$$

where $w_s$ is the scenario weight (Base=50%, Adverse=30%, Severe=20%).

This is the **IFRS 9 probability-weighted ECL** requirement — a single point
estimate that reflects multiple economic futures weighted by their likelihood.


In [ ]:
# ── ECL per scenario ───────────────────────────────────────────────────────
for scenario, params in SCENARIOS.items():
    # PD selection: Stage 1 → 12m, Stage 2/3 → lifetime
    pd_col  = f"PD_12m_{scenario}"
    pdl_col = f"PD_life_{scenario}"
    lgd_col = f"LGD_{scenario}"

    pd_used = np.where(
        full_df["Stage"] == 1,
        full_df[pd_col],    # 12-month PD for Stage 1
        full_df[pdl_col]    # Lifetime PD for Stage 2 & 3
    )

    full_df[f"ECL_{scenario}"] = pd_used * full_df[lgd_col] * full_df["EAD"]

# ── Scenario-weighted ECL ──────────────────────────────────────────────────
full_df["ECL_weighted"] = sum(
    SCENARIOS[s]["weight"] * full_df[f"ECL_{s}"]
    for s in SCENARIOS
)

print("ECL Summary (scenario-weighted, scaled UPB units):")
print(f"  Total ECL provision : {full_df['ECL_weighted'].sum():,.4f}")
print(f"  Mean ECL per loan   : {full_df['ECL_weighted'].mean():.6f}")
print(f"  ECL / EAD (coverage): {full_df['ECL_weighted'].sum() / full_df['EAD'].sum():.4f}")
print()
print("ECL by scenario:")
for s in SCENARIOS:
    print(f"  {s:8s}: total={full_df[f'ECL_{s}'].sum():,.4f} | "
          f"mean={full_df[f'ECL_{s}'].mean():.6f}")


In [ ]:
# ── ECL by Stage ───────────────────────────────────────────────────────────
print("ECL by IFRS 9 Stage (scenario-weighted):")
for s in [1, 2, 3]:
    sub = full_df[full_df["Stage"] == s]
    total_ecl = sub["ECL_weighted"].sum()
    total_ead = sub["EAD"].sum()
    n_loans   = len(sub)
    print(f"  Stage {s}: {n_loans:7,} loans | Total ECL = {total_ecl:,.4f} | "
          f"Coverage = {total_ecl/total_ead:.4f}" if total_ead > 0 else f"  Stage {s}: {n_loans} loans | EAD=0")

# ── ECL by Stage bar chart ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total ECL by stage
stage_ecl = full_df.groupby("Stage")["ECL_weighted"].sum()
axes[0].bar([f"Stage {s}" for s in stage_ecl.index],
            stage_ecl.values,
            color=[colors_stage[s] for s in stage_ecl.index], edgecolor="white")
axes[0].set_title("Total ECL by Stage"); axes[0].set_ylabel("ECL (scaled units)")

# Mean ECL per loan
stage_mean = full_df.groupby("Stage")["ECL_weighted"].mean()
axes[1].bar([f"Stage {s}" for s in stage_mean.index],
            stage_mean.values,
            color=[colors_stage[s] for s in stage_mean.index], edgecolor="white")
axes[1].set_title("Mean ECL per Loan by Stage")

# Scenario waterfall
scenarios_total = {s: full_df[f"ECL_{s}"].sum() for s in SCENARIOS}
scenarios_total["Weighted"] = full_df["ECL_weighted"].sum()
axes[2].bar(scenarios_total.keys(), scenarios_total.values(),
            color=["steelblue","darkorange","crimson","black"], edgecolor="white")
axes[2].set_title("Total ECL by Scenario"); axes[2].set_ylabel("ECL (scaled units)")

plt.suptitle("ECL Breakdown", fontsize=12)
plt.tight_layout(); plt.show()


## 8. Model Validation

### Required IFRS 9 / SR 11-7 Validation Metrics
Regulators and internal model risk management require:

| Metric | What It Tests |
|---|---|
| **ROC-AUC / KS Statistic** | Discriminatory power — can the model rank defaults above non-defaults? |
| **Brier Score** | Probabilistic accuracy — are predicted PDs close to observed rates? |
| **Calibration Plot** | Bias — does PD=5% correspond to ~5% actual default rate? |
| **Population Stability Index (PSI)** | Stability — has the score distribution shifted between dev and monitoring? |
| **Backtesting** | Does observed ECL match predicted ECL over historical windows? |


In [ ]:
# ── 8.1 Discriminatory Power ───────────────────────────────────────────────
# KS Statistic = max separation between cumulative default and non-default CDFs
from scipy.stats import ks_2samp

pd_val = PD_MODEL.predict_proba(X_val)[:, 1]
pd_def     = pd_val[y_val == 1]
pd_nondef  = pd_val[y_val == 0]

ks_stat, ks_pval = ks_2samp(pd_def, pd_nondef)
roc_auc          = roc_auc_score(y_val, pd_val)
brier            = brier_score_loss(y_val, pd_val)

print("=== Discriminatory Power ===")
print(f"  ROC-AUC  : {roc_auc:.4f}  (>0.70 acceptable, >0.75 good for mortgages)")
print(f"  KS Stat  : {ks_stat:.4f}  (>0.20 acceptable, >0.30 good)")
print(f"  Brier    : {brier:.6f}  (lower = better; <0.05 good for low-default portfolios)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_val, pd_val)
axes[0].plot(fpr, tpr, lw=2, label=f"ROC AUC = {roc_auc:.4f}")
axes[0].plot([0,1],[0,1],'k--', lw=0.8)
axes[0].fill_between(fpr, tpr, alpha=0.15)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve (Val Set)"); axes[0].legend()

# KS plot
pd_sorted = np.sort(pd_val)
cdf_def   = np.array([(pd_sorted <= p).mean() if (pd_def <= pd_sorted).any()
                       else 0 for p in pd_sorted])
cdf_nondef= np.array([(pd_sorted <= p).mean() for p in pd_sorted])

axes[1].plot(pd_sorted, np.searchsorted(np.sort(pd_def), pd_sorted) / len(pd_def),
             label="Defaults", color="crimson")
axes[1].plot(pd_sorted, np.searchsorted(np.sort(pd_nondef), pd_sorted) / len(pd_nondef),
             label="Non-Defaults", color="steelblue")
axes[1].set_title(f"KS Plot (KS={ks_stat:.4f})"); axes[1].set_xlabel("Predicted PD")
axes[1].set_ylabel("Cumulative %"); axes[1].legend()

plt.suptitle("IFRS 9 Model Validation — Discriminatory Power", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── 8.2 Population Stability Index (PSI) ──────────────────────────────────
# PSI measures whether the PD score distribution has shifted between
# development (train) and monitoring (val/test) populations.
# PSI < 0.10: stable | 0.10–0.25: monitoring required | >0.25: model rebuild

def compute_psi(base_scores, monitor_scores, n_bins=10):
    """PSI = Σ (actual% - expected%) × ln(actual% / expected%)"""
    bins = np.percentile(base_scores, np.linspace(0, 100, n_bins + 1))
    bins[0]  -= 1e-9
    bins[-1] += 1e-9

    base_pct    = np.histogram(base_scores,    bins=bins)[0] / len(base_scores)
    monitor_pct = np.histogram(monitor_scores, bins=bins)[0] / len(monitor_scores)

    # Avoid log(0)
    base_pct    = np.where(base_pct    < 1e-6, 1e-6, base_pct)
    monitor_pct = np.where(monitor_pct < 1e-6, 1e-6, monitor_pct)

    psi = np.sum((monitor_pct - base_pct) * np.log(monitor_pct / base_pct))
    return psi, base_pct, monitor_pct, bins

pd_train = PD_MODEL.predict_proba(X_train)[:, 1]
pd_test  = PD_MODEL.predict_proba(X_test)[:, 1]

psi_val,  bp_v, mp_v, bins_v = compute_psi(pd_train, pd_val)
psi_test, bp_t, mp_t, bins_t = compute_psi(pd_train, pd_test)

def psi_label(p):
    if p < 0.10: return "STABLE"
    if p < 0.25: return "MONITOR"
    return "REBUILD"

print(f"PSI (train vs val) : {psi_val:.4f}  → {psi_label(psi_val)}")
print(f"PSI (train vs test): {psi_test:.4f}  → {psi_label(psi_test)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (psi, bp, mp, bins, label) in zip(axes, [
    (psi_val,  bp_v, mp_v, bins_v, "Val"),
    (psi_test, bp_t, mp_t, bins_t, "Test")
]):
    x = range(len(bp))
    ax.bar(x, bp * 100, alpha=0.7, label="Train (base)", color="steelblue")
    ax.bar(x, mp * 100, alpha=0.7, label=f"{label} (monitor)", color="darkorange")
    ax.set_title(f"PSI (Train vs {label}) = {psi:.4f} — {psi_label(psi)}")
    ax.set_xlabel("PD Score Decile"); ax.set_ylabel("% of Population")
    ax.legend()

plt.suptitle("Population Stability Index (PSI)", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── 8.3 ECL Backtesting ────────────────────────────────────────────────────
# Backtesting compares predicted ECL to observed actual losses.
# Here we use the val set as a one-period backtest window:
#   Predicted ECL = ECL_weighted (from our model)
#   Observed Loss = EAD × LGD × actual default flag

val_idx  = full_df.index[len(train_df) : len(train_df) + len(val_df)]
val_ecl  = full_df.loc[val_idx, "ECL_weighted"]
val_ead  = full_df.loc[val_idx, "EAD"]
val_lgd  = full_df.loc[val_idx, "LGD"]
val_flag = full_df.loc[val_idx, TARGET]

# Observed loss: actual default × LGD × EAD
observed_loss_val = val_flag.values * val_lgd.values * val_ead.values

pred_total_ecl = val_ecl.sum()
obs_total_loss = observed_loss_val.sum()
coverage_ratio = pred_total_ecl / (obs_total_loss + 1e-9)

print("=== ECL Backtesting (Val Window) ===")
print(f"  Predicted ECL (scenario-weighted) : {pred_total_ecl:,.4f}")
print(f"  Observed Losses (EAD × LGD × 1def): {obs_total_loss:,.4f}")
print(f"  Coverage Ratio (Pred / Obs)        : {coverage_ratio:.4f}")
print()
if coverage_ratio < 0.80:
    print("  ⚠ UNDER-PROVISIONED — model understates losses")
elif coverage_ratio > 1.30:
    print("  ⚠ OVER-PROVISIONED  — model overstates losses")
else:
    print("  ✓ Coverage within acceptable range (0.80 – 1.30)")

# Decile backtest: predicted ECL vs observed loss by PD decile
deciles = pd.qcut(val_ecl, q=10, labels=False, duplicates="drop") + 1
bt_df = pd.DataFrame({
    "Decile": deciles.values,
    "PredECL": val_ecl.values,
    "ObsLoss": observed_loss_val
}).groupby("Decile").sum()

fig, ax = plt.subplots(figsize=(10, 4))
x = bt_df.index
ax.bar(x - 0.2, bt_df["PredECL"], width=0.4, label="Predicted ECL", color="steelblue")
ax.bar(x + 0.2, bt_df["ObsLoss"],  width=0.4, label="Observed Loss", color="crimson", alpha=0.8)
ax.set_xlabel("ECL Decile"); ax.set_ylabel("Total (scaled units)")
ax.set_title("ECL Backtesting — Predicted vs Observed by Decile (Val Set)")
ax.legend(); plt.tight_layout(); plt.show()


## 9. SHAP Interpretability (IFRS 9 Model Documentation)

IFRS 9 requires that banks document **why** the model produces the PD estimates
it does. SHAP values satisfy this requirement by providing:

1. **Global importance** — which features drive defaults most across the portfolio
2. **Directional effects** — does higher LTV increase or decrease PD?
3. **Individual loan explanation** — required for Stage 2/3 classification justification
   and for relationship manager reporting


In [ ]:
if SHAP_AVAILABLE:
    print("Computing SHAP values for PD model (RF challenger)...")

    rf_explainer = shap.TreeExplainer(rf_pd, feature_perturbation="tree_path_dependent")

    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=2000, random_state=42)
    _, idx = next(sss.split(X_val, y_val))
    X_shap = X_val.iloc[idx].reset_index(drop=True)

    shap_vals = rf_explainer.shap_values(X_shap)
    shap_default = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    plt.sca(axes[0])
    shap.summary_plot(shap_default, X_shap, plot_type="bar", show=False, plot_size=None)
    axes[0].set_title("Global Feature Importance (Mean |SHAP|)", fontsize=11)

    plt.sca(axes[1])
    shap.summary_plot(shap_default, X_shap, plot_type="dot", show=False, plot_size=None)
    axes[1].set_title("SHAP Direction & Spread", fontsize=11)

    plt.suptitle("SHAP — RF PD Model (IFRS 9 Explainability)", fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print("SHAP not installed. Run: pip install shap")


In [ ]:
if SHAP_AVAILABLE:
    # ── Individual loan explanation: highest Stage 2 risk loan ─────────────
    stage2_idx = full_df[full_df["Stage"] == 2].index
    X_full_stage2 = X_full.loc[stage2_idx]

    # Highest ECL in Stage 2
    worst_s2 = full_df.loc[stage2_idx, "ECL_weighted"].idxmax()
    worst_x  = X_full.loc[[worst_s2]]
    worst_pd = PD_MODEL.predict_proba(worst_x)[0, 1]
    worst_ecl= full_df.loc[worst_s2, "ECL_weighted"]

    print(f"Stage 2 loan with highest ECL:")
    print(f"  Predicted PD  : {worst_pd:.4f}")
    print(f"  LGD           : {full_df.loc[worst_s2, 'LGD']:.4f}")
    print(f"  EAD           : {full_df.loc[worst_s2, 'EAD']:.4f}")
    print(f"  ECL (weighted): {worst_ecl:.6f}")

    sv_single = rf_explainer.shap_values(worst_x)
    sv_default = sv_single[1][0] if isinstance(sv_single, list) else sv_single[0]
    ev = rf_explainer.expected_value
    if isinstance(ev, list): ev = ev[1]

    exp = shap.Explanation(
        values=sv_default,
        base_values=ev,
        data=worst_x.values[0],
        feature_names=worst_x.columns.tolist()
    )
    plt.figure()
    shap.waterfall_plot(exp, max_display=15, show=True)
    plt.title(f"SHAP Waterfall — Highest-Risk Stage 2 Loan (PD={worst_pd:.4f})")


## 10. Portfolio ECL Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: Stage distribution ───────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
stage_counts_plot = full_df["Stage"].value_counts().sort_index()
ax1.pie(
    stage_counts_plot.values,
    labels=[f"Stage {s}\n{stage_counts_plot[s]:,}" for s in stage_counts_plot.index],
    colors=["steelblue", "darkorange", "crimson"],
    autopct="%1.1f%%", startangle=90, pctdistance=0.75
)
ax1.set_title("Portfolio by IFRS 9 Stage")

# ── Panel 2: ECL by scenario ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ecl_totals = {s: full_df[f"ECL_{s}"].sum() for s in SCENARIOS}
ecl_totals["Weighted"] = full_df["ECL_weighted"].sum()
ax2.bar(ecl_totals.keys(), ecl_totals.values(),
        color=["steelblue", "darkorange", "crimson", "black"], edgecolor="white")
ax2.set_title("Total ECL by Scenario"); ax2.set_ylabel("ECL (scaled units)")
ax2.tick_params(axis="x", rotation=15)

# ── Panel 3: Coverage ratio by stage ─────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
for s in [1, 2, 3]:
    sub = full_df[full_df["Stage"] == s]
    total_ecl = sub["ECL_weighted"].sum()
    total_ead = sub["EAD"].sum()
    cov = total_ecl / total_ead if total_ead > 0 else 0
    ax3.bar(f"Stage {s}", cov * 100, color=colors_stage[s], edgecolor="white")
ax3.set_title("ECL Coverage Ratio by Stage"); ax3.set_ylabel("ECL / EAD (%)")

# ── Panel 4: PD distribution by stage ────────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
for s in [1, 2, 3]:
    sub = full_df[full_df["Stage"] == s]
    ax4.hist(sub["PD_12m_Base"], bins=50, alpha=0.6,
             color=colors_stage[s], label=f"Stage {s}", density=True)
ax4.set_title("PD Distribution by Stage (Base Scenario)")
ax4.set_xlabel("PD (12-month)"); ax4.set_ylabel("Density"); ax4.legend()
ax4.set_xlim(0, 0.5)

# ── Panel 5: ECL waterfall (base → adverse → severe → weighted) ───────────
ax5 = fig.add_subplot(gs[1, 2])
labels_wf = list(SCENARIOS.keys()) + ["Weighted"]
vals_wf   = [full_df[f"ECL_{s}"].sum() for s in SCENARIOS] + [full_df["ECL_weighted"].sum()]
ax5.bar(labels_wf, vals_wf, color=["steelblue","darkorange","crimson","black"], edgecolor="white")
ax5.set_title("ECL Scenario Waterfall"); ax5.tick_params(axis="x", rotation=20)

# ── Panel 6: Key metrics table ────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :])
ax6.axis("off")
metrics_table = [
    ["Metric", "Value", "Benchmark"],
    ["ROC-AUC (Val)",            f"{roc_auc:.4f}",         "> 0.70"],
    ["KS Statistic",             f"{ks_stat:.4f}",          "> 0.20"],
    ["Brier Score",              f"{brier:.6f}",            "< 0.05"],
    ["PSI (Train→Val)",          f"{psi_val:.4f}",          "< 0.10"],
    ["PSI (Train→Test)",         f"{psi_test:.4f}",         "< 0.10"],
    ["ECL Coverage Ratio",       f"{coverage_ratio:.4f}",   "0.80 – 1.30"],
    ["Stage 1 (%)",              f"{stage_pct.get(1,0):.1f}%", "—"],
    ["Stage 2 (%)",              f"{stage_pct.get(2,0):.1f}%", "—"],
    ["Stage 3 (%)",              f"{stage_pct.get(3,0):.1f}%", "—"],
    ["Total ECL (weighted)",     f"{full_df['ECL_weighted'].sum():,.4f}", "—"],
    ["Mean PD Base",             f"{full_df['PD_12m_Base'].mean():.4f}", "—"],
    ["Mean PD Severe",           f"{full_df['PD_12m_Severe'].mean():.4f}", "—"],
    ["Mean LGD",                 f"{full_df['LGD'].mean():.4f}", "—"],
]
t = ax6.table(cellText=metrics_table[1:], colLabels=metrics_table[0],
              cellLoc="center", loc="center",
              colWidths=[0.4, 0.3, 0.3])
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.4)
# Header styling
for j in range(3):
    t[(0, j)].set_facecolor("#2c3e50"); t[(0, j)].set_text_props(color="white", fontweight="bold")
ax6.set_title("IFRS 9 Model Validation & Portfolio Summary", fontsize=11, pad=12)

plt.suptitle("IFRS 9 ECL Framework — Portfolio Dashboard", fontsize=14, y=1.01)
plt.savefig("ifrs9_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard saved to ifrs9_dashboard.png")


## 11. Save Outputs

In [ ]:
# Save full portfolio with all IFRS 9 outputs
output_cols = [
    "defaulted_flag", "Stage",
    "PD_12m", "PD_lifetime",
    "PD_12m_Base", "PD_12m_Adverse", "PD_12m_Severe",
    "PD_life_Base","PD_life_Adverse","PD_life_Severe",
    "LGD", "LGD_Base","LGD_Adverse","LGD_Severe",
    "EAD",
    "ECL_Base","ECL_Adverse","ECL_Severe","ECL_weighted",
]
output_cols = [c for c in output_cols if c in full_df.columns]

ifrs9_output = full_df[output_cols].copy()
ifrs9_output.to_csv(BASE + "ifrs9_ecl_output.csv", index=False)

joblib.dump(PD_MODEL, BASE + "pd_model_calibrated.pkl")
joblib.dump(lgd_model, BASE + "lgd_model.pkl")

print(f"Saved: ifrs9_ecl_output.csv  ({ifrs9_output.shape})")
print(f"Saved: pd_model_calibrated.pkl")
print(f"Saved: lgd_model.pkl")
print()
print("=== FINAL IFRS 9 PROVISION SUMMARY ===")
for s in SCENARIOS:
    print(f"  {s:8s} ECL : {full_df[f'ECL_{s}'].sum():>12,.4f}  "
          f"(weight={SCENARIOS[s]['weight']:.0%})")
print(f"  {'Weighted':8s} ECL : {full_df['ECL_weighted'].sum():>12,.4f}")
print()
print("Stage allocation:")
for s in [1,2,3]:
    n = (full_df["Stage"]==s).sum()
    print(f"  Stage {s}: {n:,} loans ({n/len(full_df):.1%})")
